# Data Normalization Pipeline

Este notebook transforma el dataset crudo de trayectorias en tensores listos para entrenar el modelo SDE.

## Objetivo
Convertir vuelos reales (longitud variable y ruido) en secuencias limpias, normalizadas e interpoladas con longitud fija.

## Resultado final
Se generan dos archivos:
- `flight_data.pt`: lote de entrenamiento.
- `flight_data_test.pt`: un vuelo reservado para evaluación.

## Flujo del pipeline

Este proceso se ejecuta en 5 fases dentro de `master_pipeline(...)`:

1. Separación de vuelos por identificador y cortes temporales.
2. Filtro de trayectorias completas + cálculo de fuel flow.
3. Reducción de ruido y clipping de valores extremos.
4. Normalización global de variables.
5. Interpolación temporal e exportación a tensores PyTorch.

## Columnas mínimas esperadas en el CSV

- `time`
- `icao24`
- `callsign`
- `vel_mps`
- `baro_altitude`
- `track`

Si falta alguna, el pipeline no podrá construir los tensores correctamente.

In [ ]:
import pandas as pd
import numpy as np
import torch
from scipy.interpolate import interp1d
import os
import random
from openap.fuel import FuelFlow

# ==========================================
# RUTAS Y CONFIGURACION GLOBAL
# ==========================================
# CSV de entrada con trayectorias crudas.
input_csv_path = "/home/edgar/GitHub/StochasticATM/Data/Datasets/dataset_total_unido.csv"
# Carpeta destino para tensores procesados.
output_dir = "../Datasets/"
output_pt_path = os.path.join(output_dir, "flight_data.pt")
test_pt_path = os.path.join(output_dir, "flight_data_test.pt")
# Longitud fija de cada vuelo tras interpolacion temporal.
num_steps = 200


def master_pipeline(input_path, output_path, test_path):
    """Pipeline completo de limpieza, normalizacion e interpolacion.

    Entrada: CSV crudo de trayectorias.
    Salida: tensores .pt para train y test, con malla temporal comun.
    """
    print("1. Cargando dataset en bruto...")
    df = pd.read_csv(input_path)

    # Validacion minima de columnas para evitar fallos silenciosos.
    required_cols = {'time', 'icao24', 'callsign', 'vel_mps', 'baro_altitude', 'track'}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"Faltan columnas requeridas en el CSV: {sorted(missing_cols)}")

    df = df.dropna(subset=['time', 'vel_mps', 'baro_altitude', 'track'])

    # =================================================================
    # FASE 1: SEPARACION REAL DE VUELOS
    # =================================================================
    print("2. Separando vuelos por IDs y saltos de tiempo...")
    df['flight_id_base'] = df['icao24'].astype(str) + "_" + df['callsign'].astype(str)
    df = df.sort_values(by=['flight_id_base', 'time'])

    # Si hay un salto de tiempo grande, se considera nueva trayectoria.
    df['time_gap'] = df.groupby('flight_id_base')['time'].diff() > 1200
    df['unique_trajectory_id'] = df['flight_id_base'] + "_" + df.groupby('flight_id_base')['time_gap'].cumsum().astype(str)

    # =================================================================
    # FASE 2: CALCULOS FISICOS Y FILTRO DE VUELOS COMPLETOS
    # =================================================================
    print("3. Filtrando vuelos incompletos y calculando Fuel Flow...")
    fuelflow_model = FuelFlow(ac='A320')
    MASA_AVION = 65000
    cleaned_data = []

    for fid, flight in df.groupby('unique_trajectory_id'):
        f = flight.copy()

        # Ignorar trayectorias con pocos puntos.
        if len(f) < 100:
            continue

        # -------------------------------------------------------------
        # FILTRO DE TRAYECTORIA COMPLETA (despegue -> crucero -> aterrizaje)
        # -------------------------------------------------------------
        alt_inicial = f['baro_altitude'].iloc[:5].min()
        alt_final = f['baro_altitude'].iloc[-5:].min()
        alt_maxima = f['baro_altitude'].max()

        # Umbrales en metros.
        if alt_inicial > 3000 or alt_final > 3000 or alt_maxima < 6000:
            continue

        # Suavizado previo para estabilizar entrada a OpenAP.
        f['baro_altitude'] = f['baro_altitude'].rolling(window=20, min_periods=1, center=True).mean()
        f['vel_mps'] = f['vel_mps'].rolling(window=20, min_periods=1, center=True).mean()

        # Derivadas verticales limpias.
        f['dt'] = f['time'].diff().fillna(method='bfill')
        f['dalt'] = f['baro_altitude'].diff().fillna(0)
        f['vrate_mps'] = (f['dalt'] / f['dt']).clip(-25, 25)

        # Fuel flow por fila usando OpenAP.
        def calc_fuel(row):
            try:
                return fuelflow_model.enroute(
                    mass=MASA_AVION,
                    tas=row['vel_mps'] * 1.94384,
                    alt=row['baro_altitude'] * 3.28084,
                    vs=row['vrate_mps'] * 196.85
                )
            except Exception:
                # Fallback conservador si OpenAP falla en una fila puntual.
                return 0.5

        f['fuel_flow_kgs'] = f.apply(calc_fuel, axis=1)
        f = f.drop(columns=['dt', 'dalt'])
        cleaned_data.append(f)

    if not cleaned_data:
        print("ERROR: Ningun vuelo supero el filtro de trayectoria completa.")
        return

    # Consolidar vuelos validos.
    df = pd.concat(cleaned_data)

    # Geometria circular del track para evitar discontinuidad en 0/360 grados.
    df['track_sin'] = np.sin(np.radians(df['track']))
    df['track_cos'] = np.cos(np.radians(df['track']))

    # =================================================================
    # FASE 3: ESCUDO ANTI-RUIDO DE MACHINE LEARNING
    # =================================================================
    print(f"4. Aplicando filtros de ruido a {len(df['unique_trajectory_id'].unique())} vuelos completos...")

    # Clipping fisico razonable.
    df['vel_mps'] = df['vel_mps'].clip(lower=0.0, upper=340.0)
    df['baro_altitude'] = df['baro_altitude'].clip(lower=0.0, upper=14000.0)
    fuel_max_logical = df['fuel_flow_kgs'].quantile(0.999)
    df['fuel_flow_kgs'] = df['fuel_flow_kgs'].clip(lower=0.0, upper=fuel_max_logical)

    smooth_features = ['fuel_flow_kgs', 'vel_mps', 'baro_altitude', 'vrate_mps']

    # Filtro robusto a picos (mediana) + suavizado final (media).
    for feat in smooth_features:
        df[feat] = df.groupby('unique_trajectory_id')[feat].transform(
            lambda x: x.rolling(window=7, min_periods=1, center=True).median()
        )

    for feat in smooth_features:
        df[feat] = df.groupby('unique_trajectory_id')[feat].transform(
            lambda x: x.rolling(window=15, min_periods=1, center=True).mean()
        )

    # =================================================================
    # FASE 4: NORMALIZACION GLOBAL
    # =================================================================
    print("5. Normalizando datos (Escala 0 a 1)...")
    features = ['fuel_flow_kgs', 'vel_mps', 'baro_altitude', 'vrate_mps', 'track_sin', 'track_cos']
    stats = {}

    for feat in features:
        if feat in ['track_sin', 'track_cos']:
            f_min, f_max = -1.0, 1.0
        else:
            f_min, f_max = df[feat].min(), df[feat].max()

        denom = (f_max - f_min) if f_max > f_min else 1.0
        df[feat] = (df[feat] - f_min) / denom
        stats[feat] = {'min': float(f_min), 'max': float(f_max)}

    # =================================================================
    # FASE 5: CREACION DE TENSORES E INTERPOLACION
    # =================================================================
    print("6. Interpolando a 200 pasos y creando Tensores...")
    unique_trajectories = df['unique_trajectory_id'].unique()
    all_flights_data = []

    for tid in unique_trajectories:
        traj_df = df[df['unique_trajectory_id'] == tid].copy()

        t_orig = traj_df['time'].values
        t_span = t_orig.max() - t_orig.min()
        if t_span <= 0:
            # Evitar divisiones por cero en trayectorias degeneradas.
            continue

        t_norm = (t_orig - t_orig.min()) / t_span
        t_new = np.linspace(0, 1, num_steps)

        resampled_features = []
        for feat in features:
            f_interp = interp1d(
                t_norm,
                traj_df[feat].values,
                kind='linear',
                bounds_error=False,
                fill_value="extrapolate"
            )
            resampled_features.append(f_interp(t_new))

        all_flights_data.append(np.stack(resampled_features, axis=-1))

    if len(all_flights_data) < 2:
        print("ERROR: Se necesitan al menos 2 trayectorias validas para separar train/test.")
        return

    random.shuffle(all_flights_data)
    test_flight_data = all_flights_data.pop()

    # Formato final esperado por el notebook de prediccion: [time, batch, features].
    train_tensor = torch.tensor(np.stack(all_flights_data), dtype=torch.float32).permute(1, 0, 2)
    test_tensor = torch.tensor(test_flight_data, dtype=torch.float32).unsqueeze(0).permute(1, 0, 2)
    t_grid = torch.linspace(0, 1, num_steps)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    torch.save({'y_true': train_tensor, 't': t_grid, 'stats': stats}, output_path)
    torch.save({'y_true': test_tensor, 't': t_grid, 'stats': stats}, test_path)

    print("-" * 40)
    print("EXITO: Tensores de vuelos completos guardados.")
    print(f"Batch Train: {train_tensor.shape[1]} vuelos | Batch Test: 1 vuelo")
    print("-" * 40)


if __name__ == "__main__":
    # Ejecuta pipeline end-to-end usando las rutas configuradas arriba.
    master_pipeline(input_csv_path, output_pt_path, test_pt_path)

1. Cargando dataset en bruto...
2. Separando vuelos por IDs y saltos de tiempo...
3. Filtrando vuelos incompletos y calculando Fuel Flow...


/tmp/ipykernel_153504/802932380.py:69: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  f['dt'] = f['time'].diff().fillna(method='bfill')
/tmp/ipykernel_153504/802932380.py:69: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  f['dt'] = f['time'].diff().fillna(method='bfill')
/tmp/ipykernel_153504/802932380.py:69: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  f['dt'] = f['time'].diff().fillna(method='bfill')
/tmp/ipykernel_153504/802932380.py:69: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  f['dt'] = f['time'].diff().fillna(method='bfill')
/tmp/ipykernel_153504/802932380.py:69: FutureWarning: Series.fillna with 'method' is deprecated and will

4. Aplicando filtros de ruido a 128 vuelos completos...
5. Normalizando datos (Escala 0 a 1)...
6. Interpolando a 200 pasos y creando Tensores...
----------------------------------------
ÉXITO: Tensores de vuelos completos guardados.
Batch Train: 127 vuelos | Batch Test: 1 vuelo
----------------------------------------


## Salida esperada

Al terminar correctamente, deberías ver en `Data/Datasets/`:

- `flight_data.pt`: contiene `y_true`, `t` y `stats` para entrenamiento.
- `flight_data_test.pt`: mismo formato, con 1 vuelo para test.

Estructura de tensores:

- `y_true`: `[time, batch, features]`
- `t`: grilla temporal normalizada `[0, 1]`
- `stats`: min/max por feature para desnormalizar después.

## Troubleshooting rapido

- Si aparece error de columnas faltantes: revisa el CSV de entrada.
- Si no hay vuelos validos: relaja umbrales de filtro o usa más datos.
- Si OpenAP falla puntualmente: el pipeline usa un valor fallback de fuel flow.
- Si no se pueden crear train/test: asegúrate de tener al menos 2 trayectorias válidas tras limpieza.